# analysis_v4 — eval/basic/ 결과 통합 분석

`eval/basic/<machine>/<run>/` 구조를 자동 탐색하여 H100x8, RTX3090 실험 결과를
머신/모델/라우팅 단위로 비교한다.

v3 대비 추가:
- `eval/basic/` 자동 discovery (RESULTS_ROOT 로 다른 경로도 가능)
- 머신 × 모델 그룹핑 요약 테이블
- H100x8 per-GPU util 분해 (8 GPU 균등 부하 확인)
- Xeon 8480+ 2-NUMA 분할 뷰 (core 0~55 vs 56~111, v5 검증 대응)
- `wave-batch` 라우팅 색상 추가
- Run 선택 multi-filter (MACHINES / MODELS / MODES)


In [ ]:
# ============================================================
#  Configuration — 자동 탐색 대상 경로 / 필터
# ============================================================
RESULTS_ROOT = "basic"          # eval/basic/<machine>/<run>/
# 필터 (빈 set = 전체)
MACHINES = set()                # 예: {"H100x8"}, {"RTX3090"}
MODELS   = set()                # 예: {"Qwen2.5-7B-Instruct"}
MODES    = set()                # 예: {"hybrid"} / {"gpu_only"}
# 정렬 기준: "name" (dir name) | "wall" | "tokps"
SORT_BY  = "name"


In [ ]:
# ============================================================
#  Load runs — eval/basic/<machine>/<run>/ 자동 walk
# ============================================================
import json, warnings, re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': '#cccccc',
})

PALETTE = {
    'GPU-only':   '#C27900',
    'cpu-first':  '#1D4ED8',
    'gpu-first':  '#15803D',
    'RR':         '#BE185D',
    'wave-batch': '#7C3AED',   # purple
    'other':      '#4B5563',
}

def _cmap(name, light, dark):
    return LinearSegmentedColormap.from_list(name, [light, dark])

CMAPS = {
    'GPU-only':   _cmap('amber',  '#FFFBEB', '#C27900'),
    'cpu-first':  _cmap('blue',   '#EFF6FF', '#1D4ED8'),
    'gpu-first':  _cmap('green',  '#F0FDF4', '#15803D'),
    'RR':         _cmap('rose',   '#FDF2F8', '#BE185D'),
    'wave-batch': _cmap('purple', '#F5F3FF', '#7C3AED'),
    'other':      _cmap('gray',   '#F9FAFB', '#4B5563'),
}

def load_run(run_dir, machine):
    run = {'dir': run_dir, 'name': run_dir.name, 'machine': machine}
    for mode in ('hybrid', 'gpu_only'):
        jf = run_dir / f'{mode}.json'
        if jf.exists():
            with open(jf) as f: run['bench'] = json.load(f)
            run['mode'] = mode
            break
    if 'bench' not in run:
        return None
    si_path = run_dir / 'system_info.json'
    run['sysinfo'] = json.load(open(si_path)) if si_path.exists() else {}
    for mtype in ('gpu', 'cpu'):
        csv_path = run_dir / f"{run['mode']}_monitor_{mtype}.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            if 'elapsed_s' in df.columns:
                df['elapsed_s'] = df['elapsed_s'] - df['elapsed_s'].iloc[0]
            run[f'{mtype}_csv'] = df
        else:
            run[f'{mtype}_csv'] = None
    # 파생 필드
    b = run['bench']
    run['model'] = b.get('model_id','?').split('/')[-1]
    hc = run['sysinfo'].get('hybrid_config', {})
    run['hybrid_config'] = hc
    return run

def routing_label(run):
    if run['mode'] == 'gpu_only':
        return 'GPU-only'
    hc = run['hybrid_config']
    s = hc.get('routing_strategy', '?')
    p = hc.get('routing_priority', '?')
    if s == 'round-robin': return 'RR'
    if s == 'wave-batch':  return f'wave-batch({p})'
    return f'{s}({p})'

def run_color_key(r):
    lbl = routing_label(r)
    if lbl == 'GPU-only': return 'GPU-only'
    if lbl.startswith('wave-batch'): return 'wave-batch'
    if 'cpu-first' in lbl: return 'cpu-first'
    if 'gpu-first' in lbl: return 'gpu-first'
    if lbl == 'RR': return 'RR'
    return 'other'

def make_label(r):
    return f"{routing_label(r)} | {r['model']} | {r['machine']}"

# ── Walk eval/basic/ ────────────────────────────────────────
root = Path(RESULTS_ROOT)
assert root.is_dir(), f"Not found: {root.resolve()}"

runs = []
for machine_dir in sorted(root.iterdir()):
    if not machine_dir.is_dir(): continue
    machine = machine_dir.name
    if MACHINES and machine not in MACHINES: continue
    for run_dir in sorted(machine_dir.iterdir()):
        if not run_dir.is_dir(): continue
        r = load_run(run_dir, machine)
        if r is None: continue
        if MODELS and r['model'] not in MODELS: continue
        if MODES and r['mode'] not in MODES: continue
        runs.append(r)

# 정렬
if SORT_BY == 'wall':
    runs.sort(key=lambda r: r['bench'].get('wall_time_s') or r['bench'].get('duration', 0))
elif SORT_BY == 'tokps':
    runs.sort(key=lambda r: -(r['bench'].get('output_throughput') or 0))
# else name 순 (이미 sorted)

print(f'Loaded {len(runs)} runs from {root.resolve()}')
for i, r in enumerate(runs):
    print(f'  [{i}] {make_label(r):<55s}  {r["name"]}')


In [ ]:
# ============================================================
#  Section 1: 머신 × 모델 × 라우팅 그룹 요약
# ============================================================
rows = []
for i, r in enumerate(runs):
    b = r['bench']
    mon = r.get('gpu_csv')
    avg_p = mon['gpu_avg_power_w'].mean() if mon is not None and 'gpu_avg_power_w' in mon.columns else None
    tok = b.get('output_throughput', 0)
    tps_w = f'{tok/avg_p:.2f}' if avg_p and tok else '-'
    rows.append({
        '#': i,
        'Machine': r['machine'],
        'Model':   r['model'],
        'Routing': routing_label(r),
        'Wall(s)':  round(b.get('wall_time_s') or b.get('duration',0), 1),
        'Duration(s)': round(b.get('duration',0), 1),
        'Req/s':   round(b.get('request_throughput',0), 2),
        'Tok/s':   round(tok, 0),
        'TTFT(ms)': round(b.get('mean_ttft_ms',0), 0),
        'P99 TTFT': round(b.get('p99_ttft_ms',0), 0),
        'TPOT(ms)': round(b.get('mean_tpot_ms',0), 2),
        'P99 TPOT': round(b.get('p99_tpot_ms',0), 2),
        'Avg W':   f'{avg_p:.0f}' if avg_p else '-',
        'Tok/s/W': tps_w,
        'Done':    f'{b.get("completed","?")}/{b.get("num_prompts","?")}',
    })
df = pd.DataFrame(rows).set_index('#')

def color_row(row):
    key = row['Routing']
    bg = {
        'GPU-only':   '#fffde7',
    }.get(key)
    if bg is None:
        if 'cpu-first' in key:       bg = '#e3f2fd'
        elif 'gpu-first' in key:     bg = '#e8f5e9'
        elif key == 'RR':            bg = '#fce4ec'
        elif 'wave-batch' in key:    bg = '#f3e8ff'
        else:                        bg = '#f5f5f5'
    return [f'background-color: {bg}'] * len(row)

display(df.style.apply(color_row, axis=1).set_table_styles([
    {'selector':'th','props':[('background-color','#2d4a7a'),('color','white'),
                              ('font-size','12px'),('padding','5px 10px')]},
]).set_caption('<b>All Runs — machine × model × routing</b>'))


In [ ]:
# ============================================================
#  Section 2: 그룹별 best/worst (동일 Machine+Model 내 비교)
# ============================================================
if runs:
    grp = df.reset_index().groupby(['Machine','Model'])
    for (m, mdl), g in grp:
        if len(g) < 2: continue
        gs = g.sort_values('Tok/s', ascending=False)
        print(f'[{m} / {mdl}]  {len(g)} runs')
        display(gs[['#','Routing','Wall(s)','Tok/s','TTFT(ms)','TPOT(ms)','Avg W','Tok/s/W']]
                .style.set_caption(f'<b>{m} / {mdl}</b>'))
else:
    print('no runs')


In [ ]:
# ============================================================
#  Section 3: vs [0] diff (동일 machine+model 기준으로만 의미 있음)
# ============================================================
if len(runs) >= 2:
    METRICS = [
        ('wall_time_s',        'Wall(s)',    'lower', '.1f'),
        ('output_throughput',  'Tok/s',      'higher','.0f'),
        ('request_throughput', 'Req/s',      'higher','.2f'),
        ('mean_ttft_ms',       'TTFT(ms)',   'lower', '.1f'),
        ('p99_ttft_ms',        'P99 TTFT',   'lower', '.1f'),
        ('mean_tpot_ms',       'TPOT(ms)',   'lower', '.2f'),
        ('p99_tpot_ms',        'P99 TPOT',   'lower', '.2f'),
    ]
    base = runs[0]
    print(f'Base [0]: {make_label(base)}\n')
    for i in range(1, len(runs)):
        comp = runs[i]
        same = (comp['machine']==base['machine'] and comp['model']==base['model'])
        tag = '' if same else '  ⚠ 다른 machine/model'
        diff_rows = []
        for key, lbl, dirc, fmt in METRICS:
            bv = base['bench'].get(key); cv = comp['bench'].get(key)
            if not bv or cv is None: continue
            dp = (cv - bv) / abs(bv) * 100
            if dirc == 'higher':
                mk = '▲' if dp > 1 else '▼' if dp < -1 else '~'
            else:
                mk = '▼' if dp > 1 else '▲' if dp < -1 else '~'
            diff_rows.append({'Metric': lbl, '[0]': format(bv,fmt),
                              f'[{i}]': format(cv,fmt), 'Diff': f'{dp:+.1f}%', '': mk})
        if diff_rows:
            ddf = pd.DataFrame(diff_rows).set_index('Metric')
            def hl(v):
                if '▲' in str(v): return 'color:green;font-weight:bold'
                if '▼' in str(v): return 'color:red;font-weight:bold'
                return ''
            display(ddf.style.applymap(hl, subset=['']).set_caption(
                f'<b>[{i}] {make_label(comp)} vs [0]{tag}</b>'))
else:
    print('need >= 2 runs')


In [ ]:
# ============================================================
#  Section 4: Throughput / Latency Bar Chart
# ============================================================
if not runs:
    print('no runs');
else:
    labels = [f'[{i}]' for i in range(len(runs))]
    cks    = [run_color_key(r) for r in runs]
    SPECS = [
        ('request_throughput', 'req/s',  'Request TP', '.2f'),
        ('output_throughput',  'tok/s',  'Output TP',  '.0f'),
        ('mean_ttft_ms',       'ms',     'Mean TTFT',  '.0f'),
        ('mean_tpot_ms',       'ms',     'Mean TPOT',  '.2f'),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle('Throughput & Latency', fontsize=14, fontweight='bold', y=1.02)
    for ax, (k, u, t, fmt) in zip(axes, SPECS):
        vals = [r['bench'].get(k,0) for r in runs]
        bdf = pd.DataFrame({'Run': labels, 'Value': vals, 'Color': cks})
        pal = {c: PALETTE[c] for c in set(cks)}
        sns.barplot(data=bdf, x='Run', y='Value', hue='Color', palette=pal, dodge=False, ax=ax,
                    width=0.55, edgecolor='white', linewidth=1.4, legend=False)
        ax.set_title(t, fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel(u)
        mv = max(vals) if max(vals)>0 else 1
        for bar, v in zip(ax.patches, vals):
            if v > 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+mv*0.012,
                        format(v,fmt), ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_ylim(0, mv*1.2)
    patches = [mpatches.Patch(color=PALETTE[run_color_key(r)], label=f'[{i}] {make_label(r)[:55]}')
               for i, r in enumerate(runs)]
    fig.legend(handles=patches, loc='lower center', ncol=min(3,len(runs)),
               fontsize=9, bbox_to_anchor=(0.5, -0.12), framealpha=0.9)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 5: GPU avg util timeline
# ============================================================
dfs = []
for i, r in enumerate(runs):
    csv = r.get('gpu_csv')
    if csv is None or 'gpu_avg_util_pct' not in csv.columns: continue
    t = csv[['elapsed_s','gpu_avg_util_pct']].copy()
    t['Run'] = f'[{i}] {make_label(r)[:45]}'
    t['_ck'] = run_color_key(r)
    dfs.append(t)

fig, ax = plt.subplots(figsize=(16,5))
if dfs:
    for d in dfs:
        label = d['Run'].iloc[0]
        color = PALETTE[d['_ck'].iloc[0]]
        ax.plot(d['elapsed_s'].values, d['gpu_avg_util_pct'].values,
                lw=2, alpha=0.9, color=color, label=label)
        ax.fill_between(d['elapsed_s'].values, d['gpu_avg_util_pct'].values,
                        alpha=0.08, color=color)
    ax.axhline(100, color='#ef4444', ls='--', lw=0.8, alpha=0.5)
    ax.set_ylim(-2, 112)
    ax.set_title('GPU Avg Utilization Over Time', fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('GPU Util (%)')
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
else:
    ax.text(0.5,0.5,'no data',ha='center',va='center',transform=ax.transAxes)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 5b: Per-GPU 분해 (H100x8 에서 8 GPU 균등 부하 확인)
# ============================================================
for i, r in enumerate(runs):
    csv = r.get('gpu_csv')
    if csv is None: continue
    cols = sorted([c for c in csv.columns
                   if re.match(r'gpu\d+_util_pct$', c)])
    if len(cols) < 2: continue
    fig, ax = plt.subplots(figsize=(16, 3.2))
    ck = run_color_key(r)
    base = PALETTE[ck]
    cmap = plt.cm.get_cmap('tab10', len(cols))
    for j, c in enumerate(cols):
        ax.plot(csv['elapsed_s'].values, csv[c].values, lw=1.0, alpha=0.7,
                color=cmap(j), label=c.replace('_util_pct',''))
    if 'gpu_avg_util_pct' in csv.columns:
        ax.plot(csv['elapsed_s'].values, csv['gpu_avg_util_pct'].values, color='black',
                lw=1.8, ls='--', label='avg', zorder=10)
    ax.set_ylim(-2, 112)
    ax.set_title(f'[{i}] {make_label(r)[:60]} — Per-GPU Util',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Util (%)')
    ax.legend(fontsize=7, ncol=min(9, len(cols)+1), loc='upper right', framealpha=0.9)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 6: CPU avg util timeline
# ============================================================
dfs = []
for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is None or 'cpu_avg_util_pct' not in csv.columns: continue
    t = csv[['elapsed_s','cpu_avg_util_pct']].copy()
    t['Run'] = f'[{i}] {make_label(r)[:45]}'
    t['_ck'] = run_color_key(r)
    dfs.append(t)

fig, ax = plt.subplots(figsize=(16,5))
if dfs:
    for d in dfs:
        label = d['Run'].iloc[0]
        color = PALETTE[d['_ck'].iloc[0]]
        ax.plot(d['elapsed_s'].values, d['cpu_avg_util_pct'].values,
                lw=2, alpha=0.9, color=color, label=label)
        ax.fill_between(d['elapsed_s'].values, d['cpu_avg_util_pct'].values,
                        alpha=0.08, color=color)
    ax.set_ylim(-2, 112)
    ax.set_title('CPU Avg Utilization Over Time', fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('CPU Util (%)')
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
else:
    ax.text(0.5,0.5,'no data',ha='center',va='center',transform=ax.transAxes)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 6b: NUMA-split 뷰 (Xeon 8480+ 2S = 2 NUMA node)
#    node 0 = core 0..cores_per_socket-1, node 1 = 나머지
#    hybrid 시 num_cpu_engines=2 가 각 NUMA 에 1:1 bind 되는지 시각화
# ============================================================
for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is None: continue
    cpu_info = r['sysinfo'].get('cpu', {})
    sockets  = int(cpu_info.get('sockets', 1))
    if sockets < 2:
        continue
    cores_per_sock = int(cpu_info.get('cores_per_socket', 0)) * int(cpu_info.get('threads_per_core', 1))
    core_cols = sorted([c for c in csv.columns if re.match(r'core\d+_util_pct$', c)],
                       key=lambda x: int(re.findall(r'\d+', x)[0]))
    if not core_cols or cores_per_sock <= 0: continue
    half = cores_per_sock
    n0 = core_cols[:half]; n1 = core_cols[half:]
    if not n1: continue

    tv = csv['elapsed_s'].values
    avg0 = csv[n0].mean(axis=1); avg1 = csv[n1].mean(axis=1)
    fig, ax = plt.subplots(figsize=(16, 3.5))
    ck = run_color_key(r); color = PALETTE[ck]
    ax.plot(tv, avg0.values, color='#1D4ED8', lw=1.8, label=f'NUMA 0  ({len(n0)} cpu)   mean={avg0.mean():.1f}%')
    ax.fill_between(tv, avg0.values, alpha=0.12, color='#1D4ED8')
    ax.plot(tv, avg1.values, color='#15803D', lw=1.8, label=f'NUMA 1  ({len(n1)} cpu)   mean={avg1.mean():.1f}%')
    ax.fill_between(tv, avg1.values, alpha=0.12, color='#15803D')
    ax.set_ylim(-2, 112)
    ax.set_title(f'[{i}] {make_label(r)[:60]} — NUMA-split CPU Util',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Avg Util (%)')
    ax.legend(fontsize=9, loc='upper right', framealpha=0.9)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 7: GPU Power / Efficiency
# ============================================================
dfs = []
for i, r in enumerate(runs):
    csv = r.get('gpu_csv')
    if csv is None or 'gpu_avg_power_w' not in csv.columns: continue
    t = csv[['elapsed_s','gpu_avg_power_w']].copy()
    t['Run'] = f'[{i}] {make_label(r)[:45]}'
    t['_ck'] = run_color_key(r)
    dfs.append(t)
fig, ax = plt.subplots(figsize=(16,4.5))
if dfs:
    for d in dfs:
        label = d['Run'].iloc[0]
        color = PALETTE[d['_ck'].iloc[0]]
        ax.plot(d['elapsed_s'].values, d['gpu_avg_power_w'].values,
                lw=1.8, alpha=0.9, color=color, label=label)
    ax.set_title('GPU Avg Power Over Time', fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Power (W)')
    ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
else:
    ax.text(0.5,0.5,'no data',ha='center',va='center',transform=ax.transAxes)
plt.tight_layout(); plt.show()

# Efficiency bars
labels = [f'[{i}]' for i in range(len(runs))]
cks    = [run_color_key(r) for r in runs]
avg_p, tps_w, tps_wh = [], [], []
for r in runs:
    csv = r.get('gpu_csv')
    ap = csv['gpu_avg_power_w'].mean() if csv is not None and 'gpu_avg_power_w' in csv.columns else 0
    tok = r['bench'].get('output_throughput',0)
    dur = r['bench'].get('duration',0)
    tot = r['bench'].get('total_output_tokens',0)
    wh = ap*dur/3600
    avg_p.append(ap)
    tps_w.append(tok/ap if ap>0 else 0)
    tps_wh.append(tot/wh if wh>0 else 0)

SPEC = [('Avg GPU Power','W', avg_p,'.0f'),
        ('tok/s/W','tok/s·W⁻¹', tps_w,'.2f'),
        ('tok/Wh','tokens·Wh⁻¹', tps_wh,'.0f')]
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('Power & Energy Efficiency', fontweight='bold', y=1.02)
for ax,(t,u,v,f) in zip(axes, SPEC):
    bdf = pd.DataFrame({'Run': labels, 'Value': v, 'Color': cks})
    pal = {c: PALETTE[c] for c in set(cks)}
    sns.barplot(data=bdf, x='Run', y='Value', hue='Color', palette=pal, dodge=False, ax=ax,
                width=0.55, edgecolor='white', linewidth=1.4, legend=False)
    ax.set_title(t, fontweight='bold'); ax.set_xlabel(''); ax.set_ylabel(u)
    mv = max(v) if max(v)>0 else 1
    for bar, vv in zip(ax.patches, v):
        if vv>0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+mv*0.012,
                    format(vv,f), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, mv*1.2)
patches = [mpatches.Patch(color=PALETTE[run_color_key(r)], label=f'[{i}] {make_label(r)[:55]}')
           for i, r in enumerate(runs)]
fig.legend(handles=patches, loc='lower center', ncol=min(3,len(runs)),
           fontsize=9, bbox_to_anchor=(0.5, -0.12), framealpha=0.9)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 8: Per-Run CPU Core Heatmap (run 별)
# ============================================================
for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is None: continue
    core_cols = sorted([c for c in csv.columns if re.match(r'core\d+_util_pct$', c)],
                       key=lambda x: int(re.findall(r'\d+', x)[0]))
    if not core_cols: continue
    n = len(core_cols)
    fh = max(1.8, min(6.0, n*0.05 + 1.0))
    fig, ax = plt.subplots(figsize=(16, fh))
    data = csv[core_cols].values.T
    ck = run_color_key(r)
    sns.heatmap(data, ax=ax, cmap=CMAPS[ck], vmin=0, vmax=100,
                xticklabels=False, yticklabels=False,
                cbar_kws={'label':'Util (%)','shrink':0.85},
                linewidths=0, rasterized=True)
    ns = data.shape[1]
    tc = min(10, ns)
    tx = np.linspace(0, ns-1, tc, dtype=int)
    ax.set_xticks(tx+0.5)
    ax.set_xticklabels([f'{csv["elapsed_s"].iloc[p]:.0f}s' for p in tx], fontsize=9)
    ax.set_xlabel('Time (s)'); ax.set_ylabel(f'core (0–{n-1})')
    ax.set_title(f'[{i}] {make_label(r)[:60]} — CPU Core Heatmap',
                 fontweight='bold', fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 9: Per-Run 결합 뷰 (CPU avg / core heatmap / RAM)
# ============================================================
for i, r in enumerate(runs):
    csv = r.get('cpu_csv')
    if csv is None: continue
    core_cols = sorted([c for c in csv.columns if re.match(r'core\d+_util_pct$', c)],
                       key=lambda x: int(re.findall(r'\d+', x)[0]))
    has_cores = bool(core_cols)
    has_ram   = 'cpu_mem_used_mb' in csv.columns
    n_rows = 1 + int(has_cores) + int(has_ram)
    fig, axes = plt.subplots(n_rows, 1, figsize=(16, 3.3*n_rows),
                             gridspec_kw={'hspace': 0.4})
    if n_rows == 1: axes = [axes]
    ck = run_color_key(r); color = PALETTE[ck]; cmap = CMAPS[ck]
    tv = csv['elapsed_s'].values
    fig.suptitle(f'[{i}] {make_label(r)[:60]} — CPU Timeline',
                 fontweight='bold', y=1.01)
    idx = 0
    ax = axes[idx]; idx += 1
    if 'cpu_avg_util_pct' in csv.columns:
        v = csv['cpu_avg_util_pct']
        ax.plot(tv, v.values, color=color, lw=1.4)
        ax.fill_between(tv, v.values, alpha=0.18, color=color)
        ax.plot(tv, v.rolling(5, center=True).mean().values, color=color, lw=2.2, ls='--', alpha=0.7)
        ax.axhline(v.mean(), color=color, lw=1, ls=':', alpha=0.8)
        vs = v.dropna()
        ax.text(0.01,0.96, f'mean={vs.mean():.1f} max={vs.max():.1f} min={vs.min():.1f} std={vs.std():.1f}',
                transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ddd',alpha=0.85))
    ax.set_ylim(0,105); ax.set_ylabel('CPU Avg (%)'); ax.set_xlabel('Time (s)')
    ax.set_title('CPU Average Utilization', fontsize=10, fontweight='semibold')

    if has_cores:
        ax = axes[idx]; idx += 1
        n = len(core_cols)
        data = csv[core_cols].values.T
        t0, t1 = float(tv[0]), float(tv[-1])
        im = ax.imshow(data, aspect='auto', cmap=cmap, vmin=0, vmax=100,
                       interpolation='bilinear', extent=[t0,t1,n-0.5,-0.5])
        cb = plt.colorbar(im, ax=ax, fraction=0.018, pad=0.01)
        cb.set_label('Util (%)', fontsize=8)
        ax.set_xlabel('Time (s)'); ax.set_ylabel(f'core (0–{n-1})')
        ax.set_title(f'Per-Core ({n} cores)', fontsize=10, fontweight='semibold')

    if has_ram:
        ax = axes[idx]; idx += 1
        used = csv['cpu_mem_used_mb']/1024
        ax.plot(tv, used.values, color=color, lw=1.4, label='Used')
        ax.fill_between(tv, used.values, alpha=0.18, color=color)
        if 'cpu_mem_avail_mb' in csv.columns:
            ax.plot(tv, (csv['cpu_mem_avail_mb']/1024).values, color=color, lw=1.1, ls='--', alpha=0.55, label='Avail')
        m = used.dropna()
        ax.text(0.01,0.96, f'mean={m.mean():.1f}GB max={m.max():.1f}GB min={m.min():.1f}GB',
                transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='#ddd',alpha=0.85))
        ax.set_ylabel('RAM (GB)'); ax.set_xlabel('Time (s)')
        ax.set_title('RAM Usage', fontsize=10, fontweight='semibold')
        ax.legend(fontsize=8)
    sns.despine(fig=fig)
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 10: System Info per Run
# ============================================================
tbl_style = [
    {'selector':'th','props':[('background-color','#2d4a7a'),('color','white'),
                              ('font-size','12px'),('padding','5px 10px')]},
    {'selector':'tr:nth-child(even)','props':[('background-color','#f5f8ff')]},
    {'selector':'td','props':[('font-size','12px'),('padding','4px 10px')]},
]
for i, r in enumerate(runs):
    si = r.get('sysinfo', {})
    hc = si.get('hybrid_config', {})
    cpu = si.get('cpu', {})
    gpu_devs = si.get('gpu', {}).get('devices', [{}])
    gpu_name = gpu_devs[0].get('name','-') if gpu_devs else '-'
    gpu_cnt  = si.get('gpu', {}).get('count','-')
    sw = si.get('software', {})
    mem = si.get('memory', {})
    ram_total = '-'
    try:
        ram_total = f"{int(mem.get('total','0 kB').replace(' kB','').strip())/1024**2:.0f} GB"
    except Exception:
        pass
    rows = [
        ('Machine',    r['machine']),
        ('Mode',       r.get('mode','?')),
        ('CPU',        cpu.get('model_name','-')),
        ('Topology',   f"{cpu.get('sockets','-')}S × {cpu.get('cores_per_socket','-')}C × {cpu.get('threads_per_core','-')}T"),
        ('RAM',        ram_total),
        ('GPU',        f"{gpu_name} × {gpu_cnt}"),
        ('vLLM',       sw.get('vllm_version','-')),
        ('Torch',      sw.get('torch_version','-')),
    ]
    if hc:
        s = hc.get('routing_strategy','?'); p = hc.get('routing_priority','?')
        rows += [
            ('Routing',      s if s=='round-robin' else f'{s} ({p})'),
            ('CPU Max Seqs', hc.get('cpu_max_seqs','?')),
            ('CPU Engines',  hc.get('num_cpu_engines','?')),
            ('CPU Threads',  hc.get('cpu_threads','?')),
            ('CPU KV GB',    hc.get('cpu_kvcache_gb','?')),
            ('NUMA Aware',   hc.get('numa_aware','?')),
        ]
    ddf = pd.DataFrame(rows, columns=['Item','Value']).set_index('Item')
    display(ddf.style.set_table_styles(tbl_style)
            .set_caption(f'<b>[{i}] {make_label(r)}</b>  —  {r["name"]}'))
